# 📦 E-Commerce Delivery Analytics
**Question:** How often do deliveries arrive late, and how much does lateness hurt reviews?

Real mode needs the Kaggle CSVs in `data/raw/` — see README.

In [ ]:
# Choose the data mode: "sample" runs offline on the synthetic fixture,
# "real" downloads/queries the official source the first time and caches it in data/raw/.
MODE = "sample"
OUT = f"outputs/notebook-{MODE}"

import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
import load_data, analysis

Path(OUT).mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 30)

> **Sample mode** uses invented data with the real schema. Every chart is stamped *SYNTHETIC SAMPLE*. Change `MODE = "real"` above to run on the official data.

## 1 · Open DuckDB views over the CSVs

In [ ]:
con = load_data.connect(MODE)
for t in load_data.FILES: print(t, con.execute(f'select count(*) from {t}').fetchone()[0])

## 2 · Check the grain of every table

In [ ]:
con.execute(analysis.GRAIN_SQL).df().T

## 3 · The trap: join first, count later

In [ ]:
print(analysis.NAIVE_SQL)
con.execute(analysis.NAIVE_SQL).df()

## 4 · Aggregate children to one row per order, then join

In [ ]:
print(analysis.ORDER_FACTS_SQL)
facts = analysis.build_order_facts(con)
print(len(facts), 'rows,', facts['order_id'].nunique(), 'distinct orders')
facts.head()

## 5 · KPIs

In [ ]:
pd.Series(analysis.kpis(con)).round(3)

## 6 · Monthly trend and reviews

In [ ]:
monthly = con.execute(analysis.MONTHLY_SQL).df()
reviews = con.execute(analysis.REVIEW_BY_LATENESS_SQL).df()
display(Image(f"{OUT}/{analysis.plot_monthly(monthly, Path(OUT), MODE == 'sample')}"))
display(Image(f"{OUT}/{analysis.plot_reviews(reviews, Path(OUT), MODE == 'sample')}"))

## 7 · Limitations and license
- Reviews can repeat per order; the first/last months are sparse.
- **CC BY-NC-SA 4.0** — non-commercial, share-alike, credit Olist; don't commit the raw CSVs.